In [ ]:
# ============================================================
# 🧩 STEP 1: Setup environment
# ============================================================
!pip install -q transformers accelerate datasets evaluate rouge-score bert-score torch torchvision torchaudio bitsandbytes

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_Token"), add_to_git_credential=True)

import torch
import time
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import evaluate
import re
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# 🧩 STEP 2: Load GSM8K dataset (100 samples)
# ============================================================
dataset = load_dataset("openai/gsm8k", "main")
test_data = dataset["test"].select(range(100))  # first 100 rows

# ============================================================
# 🧩 STEP 3: Regex for numeric extraction
# ============================================================
def extract_numeric_answer(text):
    match = re.search(r'-?\d+\.?\d*', text)
    if match:
        return match.group(0)
    return text.strip()

# ============================================================
# 🧩 STEP 4: Metrics functions
# ============================================================
def calculate_accuracy_numeric(preds, refs):
    correct = 0
    for p, r in zip(preds, refs):
        try:
            gt = re.search(r'-?\d+\.?\d*', r.split("####")[-1]).group(0)
            pred_num = extract_numeric_answer(p)
            if gt == pred_num:
                correct += 1
        except:
            continue
    return correct / len(preds)

def calculate_exact_match_numeric(preds, refs):
    matches = 0
    for p, r in zip(preds, refs):
        try:
            gt = re.search(r'-?\d+\.?\d*', r.split("####")[-1]).group(0)
            pred_num = extract_numeric_answer(p)
            if gt == pred_num:
                matches += 1
        except:
            continue
    return matches / len(preds)

def compute_metrics(preds, refs):
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")

    results = {}
    # ROUGE
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    results.update({f"ROUGE_{k}": v for k, v in rouge_scores.items()})

    # BERTScore
    bert_scores = bertscore.compute(predictions=preds, references=refs, lang="en")
    results["BERTScore_F1"] = sum(bert_scores["f1"]) / len(bert_scores["f1"])

    # Numeric Exact Match
    results["Exact_Match"] = calculate_exact_match_numeric(preds, refs)

    return results

# ============================================================
# 🧩 STEP 5: Batched Inference function with padding fix
# ============================================================
def run_inference_batched(model_name, dataset, num_samples=100, batch_size=8):
    torch.cuda.empty_cache()
    gc.collect()

    print(f"\n🚀 Running inference for: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # FIX: padding token if not present
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    model.eval()

    dataset_subset = dataset.select(range(num_samples))
    prompt_prefix = "Provide ONLY the one-word numeric answer (no punctuation, no explanation).\nQuestion: "
    inputs = [f"{prompt_prefix}{item['question']}\nOne-word Answer:" for item in dataset_subset]
    refs = [item['answer'] for item in dataset_subset]
    preds = []

    start_time = time.time()

    for i in tqdm(range(0, len(inputs), batch_size), desc="Generating answers (batched)"):
        batch_inputs = inputs[i:i+batch_size]
        tokenized = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            output = model.generate(**tokenized, max_new_tokens=64)
        for out in output:
            pred = tokenizer.decode(out, skip_special_tokens=True)
            preds.append(extract_numeric_answer(pred))

    inference_time = time.time() - start_time

    # Compute metrics
    results = compute_metrics(preds, refs)
    results["Accuracy"] = calculate_accuracy_numeric(preds, refs)
    results["Inference_Time(s)"] = round(inference_time, 2)

    # Memory & model size
    gpu_mem = torch.cuda.max_memory_allocated(device) / (1024**3)
    model_size_gb = sum(p.numel() for p in model.parameters()) * 2 / (1024**3)  # fp16
    results["Model_Size(GB)"] = round(model_size_gb, 2)
    results["Peak_GPU_Memory(GB)"] = round(gpu_mem, 2)

    # Save CSV locally
    df_model = pd.DataFrame([results])
    file_name = f"{model_name.replace('/', '_')}_results.csv"
    df_model.to_csv(file_name, index=False)
    print(f"✅ Saved results for {model_name} to {file_name}")

    del model, tokenized, output
    torch.cuda.empty_cache()
    gc.collect()

    return results

# ============================================================
# 🧩 STEP 6: Run only public LLaMA model
# ============================================================
llama_models = [
    "meta-llama/Llama-3.2-1B-Instruct"
]

llama_results = {}
for m in llama_models:
    batch_size = 8  # safe for T4 GPU
    llama_results[m] = run_inference_batched(m, test_data, num_samples=100, batch_size=batch_size)

# ============================================================
# 🧩 STEP 7: Display results
# ============================================================
df_llama = pd.DataFrame(llama_results).T
print("\n📊 Baseline Vanilla Inference Results (100 GSM8K samples) for LLaMA 3.2-1B:")
display(df_llama)


Device: cuda

🚀 Running inference for: meta-llama/Llama-3.2-1B-Instruct


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Saved results for meta-llama/Llama-3.2-1B-Instruct to meta-llama_Llama-3.2-1B-Instruct_results.csv

📊 Baseline Vanilla Inference Results (100 GSM8K samples) for LLaMA 3.2-1B:


,ROUGE_rouge1,ROUGE_rouge2,ROUGE_rougeL,ROUGE_rougeLsum,BERTScore_F1,Exact_Match,Accuracy,Inference_Time(s),Model_Size(GB),Peak_GPU_Memory(GB)
meta-llama/Llama-3.2-1B-Instruct,0.045889,0.004632,0.044375,0.044853,0.778069,0.02,0.02,26.62,2.3,3.92


In [ ]:
# ============================================================
# 🧩 STEP 1: Environment Setup
# ============================================================
!pip install -q transformers accelerate datasets evaluate rouge-score bert-score bitsandbytes torch torchvision torchaudio

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_Token"), add_to_git_credential=True)

from huggingface_hub import login
login(new_session=False)

import os
import gc
import re
import time
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # prevent fragmentation
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# 🧩 STEP 2: Load GSM8K dataset (first 100 samples)
# ============================================================
dataset = load_dataset("openai/gsm8k", "main")
test_data = dataset["test"].select(range(100))

# ============================================================
# 🧩 STEP 3: Regex and metrics helper functions
# ============================================================
def extract_numeric_answer(text):
    match = re.search(r'-?\d+\.?\d*', text)
    if match:
        return match.group(0)
    return text.strip()

def calculate_accuracy_numeric(preds, refs):
    correct = 0
    for p, r in zip(preds, refs):
        try:
            gt = re.search(r'-?\d+\.?\d*', r.split("####")[-1]).group(0)
            pred_num = extract_numeric_answer(p)
            if gt == pred_num:
                correct += 1
        except:
            continue
    return correct / len(preds)

def calculate_exact_match_numeric(preds, refs):
    matches = 0
    for p, r in zip(preds, refs):
        try:
            gt = re.search(r'-?\d+\.?\d*', r.split("####")[-1]).group(0)
            pred_num = extract_numeric_answer(p)
            if gt == pred_num:
                matches += 1
        except:
            continue
    return matches / len(preds)

def compute_metrics(preds, refs):
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")
    results = {}
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    results.update({f"ROUGE_{k}": v for k, v in rouge_scores.items()})
    bert_scores = bertscore.compute(predictions=preds, references=refs, lang="en")
    results["BERTScore_F1"] = sum(bert_scores["f1"]) / len(bert_scores["f1"])
    results["Exact_Match"] = calculate_exact_match_numeric(preds, refs)
    return results

# ============================================================
# 🧩 STEP 4: Batched inference with 4-bit support
# ============================================================
def run_inference_batched(model_name, dataset, num_samples=100, batch_size=8, quantize_4bit=False):
    torch.cuda.empty_cache()
    gc.collect()

    print(f"\n🚀 Running inference for: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16,
        "low_cpu_mem_usage": True
    }

    # enable 4-bit quantization for large models
    if quantize_4bit:
        model_kwargs["load_in_4bit"] = True

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    model.eval()

    dataset_subset = dataset.select(range(num_samples))
    prompt_prefix = "Provide ONLY the one-word numeric answer (no punctuation, no explanation).\nQuestion: "
    inputs = [f"{prompt_prefix}{item['question']}\nOne-word Answer:" for item in dataset_subset]
    refs = [item['answer'] for item in dataset_subset]
    preds = []

    start_time = time.time()

    for i in tqdm(range(0, len(inputs), batch_size), desc="Generating answers (batched)"):
        batch_inputs = inputs[i:i+batch_size]
        tokenized = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            output = model.generate(**tokenized, max_new_tokens=64)
        for out in output:
            pred = tokenizer.decode(out, skip_special_tokens=True)
            preds.append(extract_numeric_answer(pred))

    inference_time = time.time() - start_time
    results = compute_metrics(preds, refs)
    results["Accuracy"] = calculate_accuracy_numeric(preds, refs)
    results["Inference_Time(s)"] = round(inference_time, 2)
    gpu_mem = torch.cuda.max_memory_allocated(device) / (1024**3)
    model_size_gb = sum(p.numel() for p in model.parameters()) * 2 / (1024**3)
    results["Model_Size(GB)"] = round(model_size_gb, 2)
    results["Peak_GPU_Memory(GB)"] = round(gpu_mem, 2)

    df_model = pd.DataFrame([results])
    file_name = f"{model_name.replace('/', '_')}_results.csv"
    df_model.to_csv(file_name, index=False)
    print(f"✅ Saved results for {model_name} to {file_name}")

    del model, tokenized, output
    torch.cuda.empty_cache()
    gc.collect()
    return results

# ============================================================
# 🧩 STEP 5: Run both LLaMA models (OOM safe)
# ============================================================
llama_models = [
    ("meta-llama/Llama-3.2-1B-Instruct", False),  # no quantization
    ("meta-llama/Llama-2-7b-hf", True)            # quantize to 4-bit
]

llama_results = {}
for model_name, quantize in llama_models:
    batch_size = 8 if "1B" in model_name else 1
    llama_results[model_name] = run_inference_batched(
        model_name, test_data, num_samples=100,
        batch_size=batch_size, quantize_4bit=quantize
    )

# ============================================================
# 🧩 STEP 6: Display Results
# ============================================================
df_llama = pd.DataFrame(llama_results).T
print("\n📊 Baseline Vanilla Inference Results (100 GSM8K samples) for LLaMA models:")
display(df_llama)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 11.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Device: cuda


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Generating answers (batched): 100%|██████████| 100/100 [09:33<00:00,  5.73s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Saved results for meta-llama/Llama-2-7b-hf to meta-llama_Llama-2-7b-hf_results.csv

📊 Baseline Vanilla Inference Results (100 GSM8K samples) for LLaMA models:


,ROUGE_rouge1,ROUGE_rouge2,ROUGE_rougeL,ROUGE_rougeLsum,BERTScore_F1,Exact_Match,Accuracy,Inference_Time(s),Model_Size(GB),Peak_GPU_Memory(GB)
meta-llama/Llama-3.2-1B-Instruct,0.038645,0.000933,0.038627,0.038599,0.777469,0.02,0.02,24.75,2.30,3.92
meta-llama/Llama-2-7b-hf,0.044088,0.003930,0.042773,0.043355,0.777793,0.02,0.02,573.41,6.52,6.51
